# Clasificador Clumpy vs No-Clumpy con ConvNeXt-Tiny

Clasificacion binaria de galaxias clumpy usando imagenes JWST y transfer learning con ConvNeXt-Tiny.

| Parametro | Valor |
|---|---|
| Arquitectura | ConvNeXt-Tiny (pre-entrenado ImageNet) |
| Estrategia | Congelar backbone 5 epocas -> fine-tuning completo |
| Validacion | 5-Fold Stratified Cross Validation |
| Balance | WeightedRandomSampler + Focal Loss |
| Epocas | 50 con early stopping (patience=10) |
| Umbral | Optimizado por fold en val set |
| Modelo final | Retrain en todos los datos tras CV |

In [ ]:
!pip install timm -q

import os
import json
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast

import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, precision_recall_curve, average_precision_score)

import timm
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'PyTorch: {torch.__version__}')
print(f'timm: {timm.__version__}')

## 1. Montar Google Drive y verificar GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ProyectoClumps/data'
JSON_PATH = os.path.join(DATA_DIR, '11_05_2026_dictionary_with_all_statmorph_and_classifications.json')
IMAGES_DIR = os.path.join(DATA_DIR, 'colour_images')
SAVE_DIR = '/content/drive/MyDrive/ProyectoClumps'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'JSON existe: {os.path.exists(JSON_PATH)}')
print(f'Images dir existe: {os.path.exists(IMAGES_DIR)}')

## 2. Cargar datos (todos, sin split todavia)

In [ ]:
with open(JSON_PATH) as f:
    data = json.load(f)

galaxies = data['galaxies']
img_files = {os.path.splitext(f)[0] for f in os.listdir(IMAGES_DIR) if f.endswith('.png')}

samples = []
for g in galaxies:
    name = g['name']
    if name in img_files:
        label = 1 if g['info']['visual_clas']['Clumpy'] else 0
        samples.append({
            'name': name,
            'path': os.path.join(IMAGES_DIR, name + '.png'),
            'label': label
        })

df = pd.DataFrame(samples)
print(f'Total muestras con imagen: {len(df)}')
print(f'Clumpy: {(df["label"]==1).sum()} ({100*(df["label"]==1).mean():.1f}%)')
print(f'No clumpy: {(df["label"]==0).sum()} ({100*(df["label"]==0).mean():.1f}%)')

all_labels = np.array([s['label'] for s in samples])
class_counts = np.bincount(all_labels)
print(f'\nClass counts: no-clumpy={class_counts[0]}, clumpy={class_counts[1]}')

## 3. Dataset, transforms y DataLoaders

Las transforms se definen una vez. Los DataLoaders se construyen por fold.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ClumpyDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, s['label'], s['name']


def make_loaders(train_samples, val_samples, batch_size=64):
    train_labels = [s['label'] for s in train_samples]
    cc = np.bincount(train_labels)
    cw = 1. / cc
    sw = np.array([cw[l] for l in train_labels])
    sampler = WeightedRandomSampler(sw, num_samples=len(train_samples), replacement=True)

    train_ds = ClumpyDataset(train_samples, transform=train_transform)
    val_ds = ClumpyDataset(val_samples, transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    return train_loader, val_loader

print('Transforms y dataset listos')

## 4. Visualizar ejemplos del dataset

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
clumpy_shown = [s for s in samples if s['label'] == 1][:5]
non_clumpy_shown = [s for s in samples if s['label'] == 0][:5]

for col, s in enumerate(clumpy_shown):
    img = Image.open(s['path']).convert('RGB')
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Clumpy\n{s["name"]}', fontsize=9)
    axes[0, col].axis('off')

for col, s in enumerate(non_clumpy_shown):
    img = Image.open(s['path']).convert('RGB')
    axes[1, col].imshow(img)
    axes[1, col].set_title(f'No Clumpy\n{s["name"]}', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Ejemplos del dataset', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Focal Loss

Reemplaza BCEWithLogitsLoss. Reduce el peso de los ejemplos faciles (no-clumpy)
y obliga al modelo a enfocarse en los dificiles. Formula:

$$FL(p_t) = -\alpha_t (1 - p_t)^{\gamma} \log(p_t)$$

- `gamma=2`: penaliza los ejemplos bien clasificados (faciles)
- `alpha=0.9`: peso para la clase positiva (clumpy)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.9, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        logits = logits.view(-1)
        targets = targets.view(-1).float()
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        loss = focal_weight * bce
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

criterion = FocalLoss(alpha=0.9, gamma=2.0).to(device)
print('Focal Loss: alpha=0.9, gamma=2.0')

## 6. Funciones auxiliares: modelo, entrenamiento, umbral optimo

In [ ]:
def build_model(pretrained=True):
    model = timm.create_model('convnext_tiny', pretrained=pretrained, num_classes=1)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True
    return model.to(device)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, labels, _ in tqdm(loader, desc='Train', leave=False):
        images = images.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels, _ in tqdm(loader, desc='Eval', leave=False):
        images = images.to(device)
        labels = labels.float().to(device)
        with autocast('cuda'):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_labels)


def find_best_threshold(labels, probs):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_th, best_f1 = 0.5, 0.0
    for th in thresholds:
        preds = (probs > th).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th, best_f1


print('Funciones auxiliares listas')

## 7. Funcion de entrenamiento completa por fold

Encapsula el ciclo: freeze 5 epocas -> unfreeze -> early stopping.

In [ ]:
EPOCHS = 50
FREEZE_EPOCHS = 5
PATIENCE = 10
LR_HEAD = 2e-4
LR_FINETUNE = 5e-5
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 64


def train_fold(train_samples, val_samples, fold_name='fold'):
    train_loader, val_loader = make_loaders(train_samples, val_samples, batch_size=BATCH_SIZE)

    model = build_model(pretrained=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler('cuda')

    best_val_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            print(f'  [{fold_name}] Descongelando backbone en epoca {epoch}')
            for param in model.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS)
            scaler = GradScaler('cuda')

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_probs, val_labels = evaluate(model, val_loader)

        val_th, val_f1_th = find_best_threshold(val_labels, val_probs)
        val_preds = (val_probs > val_th).astype(int)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = 0.5

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        print(f'  [{fold_name}] Epoch {epoch+1}/{EPOCHS} | TrainL: {train_loss:.4f} | '
              f'ValL: {val_loss:.4f} | F1: {val_f1:.4f} (th={val_th:.2f}) | AUC: {val_auc:.4f}')

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = val_th
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  [{fold_name}] Early stopping en epoca {epoch+1}')
                break

    model.load_state_dict(best_state)
    print(f'  [{fold_name}] Mejor F1: {best_val_f1:.4f} | Umbral: {best_threshold:.3f}')
    return model, best_threshold, best_val_f1, history

## 8. 5-Fold Cross Validation

Para cada fold: entrena en 4/5 de los datos, valida en 1/5.
Guarda metricas y umbral optimo de cada fold.

In [ ]:
K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
cv_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(samples)), all_labels)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{K_FOLDS}')
    print(f'{"="*60}')

    fold_train = [samples[i] for i in train_idx]
    fold_val = [samples[i] for i in val_idx]
    n_clumpy_train = sum(s['label'] for s in fold_train)
    n_clumpy_val = sum(s['label'] for s in fold_val)
    print(f'Train: {len(fold_train)} (Clumpy: {n_clumpy_train}) | '
          f'Val: {len(fold_val)} (Clumpy: {n_clumpy_val})')

    model, best_th, best_f1, history = train_fold(fold_train, fold_val, fold_name=f'F{fold+1}')
    cv_histories.append(history)

    val_loss, val_probs, val_labels = evaluate(model, DataLoader(
        ClumpyDataset(fold_val, transform=eval_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True))
    val_preds = (val_probs > best_th).astype(int)

    fold_metrics = {
        'fold': fold + 1,
        'threshold': best_th,
        'accuracy': accuracy_score(val_labels, val_preds),
        'precision': precision_score(val_labels, val_preds, zero_division=0),
        'recall': recall_score(val_labels, val_preds, zero_division=0),
        'f1': f1_score(val_labels, val_preds, zero_division=0),
        'roc_auc': roc_auc_score(val_labels, val_probs),
        'pr_auc': average_precision_score(val_labels, val_probs),
        'val_probs': val_probs,
        'val_labels': val_labels,
    }
    cv_results.append(fold_metrics)
    print(f'  -> F1: {fold_metrics["f1"]:.4f} | Prec: {fold_metrics["precision"]:.4f} | '
          f'Rec: {fold_metrics["recall"]:.4f} | AUC: {fold_metrics["roc_auc"]:.4f}')

    del model
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('CROSS VALIDATION COMPLETADA')
print(f'{"="*60}')

## 9. Resultados de Cross Validation

In [ ]:
cv_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('val_probs', 'val_labels')}
                       for r in cv_results])
print(cv_df.to_string(index=False))

print(f'\n{"="*40}')
print('   MEDIA +/- STD (5 folds)')
print(f'{"="*40}')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
    mean = cv_df[metric].mean()
    std = cv_df[metric].std()
    print(f'{metric:>10}: {mean:.4f} +/- {std:.4f}')
print(f'{"threshold":>10}: {cv_df["threshold"].mean():.4f} +/- {cv_df["threshold"].std():.4f}')
print(f'{"="*40}')

avg_threshold = cv_df['threshold'].mean()
print(f'\nUmbral promedio para modelo final: {avg_threshold:.4f}')

## 10. Visualizaciones de CV

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics_to_plot = ['f1', 'precision', 'recall', 'roc_auc', 'pr_auc', 'threshold']
colors = ['green', 'blue', 'orange', 'red', 'purple', 'brown']

for ax, metric, color in zip(axes.flat, metrics_to_plot, colors):
    values = cv_df[metric].values
    ax.bar(range(1, K_FOLDS+1), values, color=color, alpha=0.7)
    ax.axhline(values.mean(), color='black', linestyle='--', label=f'Mean: {values.mean():.3f}')
    ax.set_title(metric.upper())
    ax.set_xlabel('Fold')
    ax.set_xticks(range(1, K_FOLDS+1))
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Metricas por Fold (5-Fold CV)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cv_metrics.png'), dpi=150)
plt.show()

In [ ]:
all_cv_probs = np.concatenate([r['val_probs'] for r in cv_results])
all_cv_labels = np.concatenate([r['val_labels'] for r in cv_results])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fpr, tpr, _ = roc_curve(all_cv_labels, all_cv_probs)
auc_all = roc_auc_score(all_cv_labels, all_cv_probs)
axes[0].plot(fpr, tpr, label=f'ROC (AUC={auc_all:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve (pooled CV)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(True)

prec_arr, rec_arr, _ = precision_recall_curve(all_cv_labels, all_cv_probs)
ap_all = average_precision_score(all_cv_labels, all_cv_probs)
axes[1].plot(rec_arr, prec_arr, label=f'PR (AP={ap_all:.3f})', color='purple')
axes[1].set_title('PR Curve (pooled CV)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(True)

all_preds_pooled = (all_cv_probs > avg_threshold).astype(int)
cm_pooled = confusion_matrix(all_cv_labels, all_preds_pooled)
sns.heatmap(cm_pooled, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Clumpy', 'Clumpy'],
            yticklabels=['No Clumpy', 'Clumpy'])
axes[2].set_title(f'Confusion Matrix (th={avg_threshold:.3f})')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cv_curves.png'), dpi=150)
plt.show()

## 11. Retrain: modelo final en TODOS los datos

Tras CV, entrenamos un modelo final usando todas las galaxias disponibles.
Usamos el umbral promedio de los 5 folds.

In [ ]:
print('Retrain final en todos los datos...')
print(f'Umbral a usar: {avg_threshold:.4f}')

final_train_samples = samples
final_val_samples = random.sample(samples, min(150, len(samples)))

final_model, final_th, final_f1, final_history = train_fold(
    final_train_samples, final_val_samples, fold_name='FINAL'
)
print(f'\nModelo final entrenado. F1 val: {final_f1:.4f}')

## 12. Grad-CAM con modelo final

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate(self, input_tensor, target_class=0):
        self.model.eval()
        output = self.model(input_tensor)
        self.model.zero_grad()
        output[0, target_class].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()


grad_cam = GradCAM(final_model, final_model.stages[3])

clumpy_examples = [s for s in samples if s['label'] == 1][:4]
non_clumpy_examples = [s for s in samples if s['label'] == 0][:4]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, s in enumerate(clumpy_examples):
    img = Image.open(s['path']).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)
    cam = grad_cam.generate(inp, target_class=0)
    axes[0, col].imshow(img.resize((224, 224)))
    axes[0, col].imshow(cam, cmap='jet', alpha=0.4)
    axes[0, col].set_title(f'Clumpy\n{s["name"]}', fontsize=9)
    axes[0, col].axis('off')
for col, s in enumerate(non_clumpy_examples):
    img = Image.open(s['path']).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)
    cam = grad_cam.generate(inp, target_class=0)
    axes[1, col].imshow(img.resize((224, 224)))
    axes[1, col].imshow(cam, cmap='jet', alpha=0.4)
    axes[1, col].set_title(f'No Clumpy\n{s["name"]}', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Grad-CAM: Activaciones para clase Clumpy', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gradcam_examples.png'), dpi=150)
plt.show()
grad_cam.remove_hooks()

## 13. Exportar modelo final

In [ ]:
final_path = os.path.join(SAVE_DIR, 'clumpy_classifier_final.pth')
torch.save({
    'model_state_dict': final_model.state_dict(),
    'model_name': 'convnext_tiny',
    'num_classes': 1,
    'threshold': float(avg_threshold),
    'final_val_f1': float(final_f1),
    'cv_metrics': {k: [float(x) for x in v] for k, v in cv_df.to_dict('list').items()},
    'cv_mean_f1': float(cv_df['f1'].mean()),
    'cv_std_f1': float(cv_df['f1'].std()),
    'final_history': {k: [float(x) for x in v] for k, v in final_history.items()},
}, final_path)

print(f'Modelo guardado en: {final_path}')
print(f'Umbral optimo: {avg_threshold:.4f}')
print(f'CV F1: {cv_df["f1"].mean():.4f} +/- {cv_df["f1"].std():.4f}')
print(f'CV ROC-AUC: {cv_df["roc_auc"].mean():.4f} +/- {cv_df["roc_auc"].std():.4f}')

## 14. Funcion de inferencia

Para predecir nuevas galaxias con el modelo final.

In [ ]:
def predict_clumpy(image_path, model=None, threshold=None):
    if model is None:
        model = build_model(pretrained=False)
        ckpt = torch.load(final_path, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'])
        model.to(device).eval()
        threshold = ckpt['threshold']
    if threshold is None:
        threshold = avg_threshold

    img = Image.open(image_path).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad(), autocast('cuda'):
        logit = model(inp).squeeze(1)
        prob = torch.sigmoid(logit).item()
    pred = 1 if prob > threshold else 0
    label = 'Clumpy' if pred == 1 else 'No Clumpy'
    return {'prediction': label, 'probability': prob, 'threshold': threshold}


test_pred = predict_clumpy(samples[0]['path'])
print(f'Ejemplo: {samples[0]["name"]}')
print(f'Real: {"Clumpy" if samples[0]["label"]==1 else "No Clumpy"}')
print(f'Predicho: {test_pred["prediction"]} (prob={test_pred["probability"]:.4f})')